In [1]:
import pandas as pd

ratings = pd.read_csv(
    "data/ratings.dat",
    sep="::",
    engine="python",
    names=["user_id", "movie_id", "rating", "timestamp"],
)

In [2]:
ratings.shape

(1000209, 4)

In [3]:
ratings.head()

,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [4]:
ratings["user_id"].nunique()

6040

In [5]:
ratings["movie_id"].nunique()

3706

In [6]:
ratings["date"] = pd.to_datetime(ratings["timestamp"],unit="s")

In [7]:
ratings.head()

,user_id,movie_id,rating,timestamp,date
0,1,1193,5,978300760,2000-12-31 22:12:40
1,1,661,3,978302109,2000-12-31 22:35:09
2,1,914,3,978301968,2000-12-31 22:32:48
3,1,3408,4,978300275,2000-12-31 22:04:35
4,1,2355,5,978824291,2001-01-06 23:38:11


In [8]:
ratings["date"].min() , ratings["date"].max()

(Timestamp('2000-04-25 23:05:32'), Timestamp('2003-02-28 17:49:50'))

In [9]:
ratings["date"].dt.to_period("M").value_counts().sort_index()

date
2000-04     11396
2000-05     67437
2000-06     54486
2000-07     90334
2000-08    182109
2000-09     52421
2000-10     42294
2000-11    290793
2000-12    113487
2001-01     18004
2001-02      8136
2001-03      6083
2001-04      5171
2001-05      4939
2001-06      4981
2001-07      4765
2001-08      4454
2001-09      3077
2001-10      2192
2001-11      2760
2001-12      3496
2002-01      3216
2002-02      2496
2002-03      2454
2002-04      2840
2002-05      1902
2002-06      1643
2002-07      1905
2002-08      2111
2002-09      1293
2002-10      1014
2002-11      1908
2002-12      1264
2003-01      1852
2003-02      1496
Freq: M, Name: count, dtype: int64

In [10]:
before_2001 = (ratings["date"] < "2001-01-01").sum()
before_2001, before_2001 / len(ratings)

(np.int64(904757), np.float64(0.9045679452994324))

In [11]:
pd.to_datetime(ratings["timestamp"].quantile(0.8), unit="s")

Timestamp('2000-12-02 14:52:18')

In [12]:
cutoff = pd.to_datetime(ratings["timestamp"].quantile(0.8), unit="s")

train = ratings[ratings["date"] < cutoff]
test = ratings[ratings["date"] >= cutoff]

In [13]:
len(train), len(test), len(train) + len(test)

(800164, 200045, 1000209)

In [14]:
popularity = train["movie_id"].value_counts()

In [15]:
popularity.head(10)

movie_id
2858    2902
260     2516
1196    2516
1210    2457
589     2284
2028    2245
480     2234
1270    2184
2571    2172
1580    2157
Name: count, dtype: int64

In [16]:
top_10 = popularity.head(10).index.tolist()
top_10

[2858, 260, 1196, 1210, 589, 2028, 480, 1270, 2571, 1580]

In [17]:
movies = pd.read_csv(
    "data/movies.dat",
    sep="::",
    engine="python",
    names=["movie_id", "title", "genres"],
    encoding="latin-1",
)

In [18]:
movies.head()

,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [19]:
movies[movies["movie_id"].isin(top_10)]

,movie_id,title,genres
257,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Fantasy|Sci-Fi
476,480,Jurassic Park (1993),Action|Adventure|Sci-Fi
585,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi|Thriller
1178,1196,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Drama|Sci-Fi|War
1192,1210,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Romance|Sci-Fi|War
1250,1270,Back to the Future (1985),Comedy|Sci-Fi
1539,1580,Men in Black (1997),Action|Adventure|Comedy|Sci-Fi
1959,2028,Saving Private Ryan (1998),Action|Drama|War
2502,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller
2789,2858,American Beauty (1999),Comedy|Drama


In [20]:
test_likes = test[test["rating"] >= 4]

In [21]:
len(test_likes)

112264

In [22]:
liked_by_user = test_likes.groupby("user_id")["movie_id"].apply(set)

In [23]:
liked_by_user

user_id
1       {1, 2692, 260, 1028, 1287, 1029, 1545, 1035, 5...
2       {1537, 515, 3334, 648, 1544, 265, 2571, 3468, ...
3       {260, 1291, 653, 1304, 1049, 2081, 2470, 552, ...
4       {480, 2366, 1954, 2947, 260, 2692, 2951, 1097,...
5       {2560, 515, 3083, 2571, 2580, 1046, 29, 32, 34...
                              ...                        
6001    {481, 965, 3751, 2600, 457, 2346, 3947, 3147, ...
6002    {2946, 2819, 2947, 2948, 2949, 1927, 1419, 909...
6016    {930, 3685, 1639, 3245, 339, 3894, 3129, 3834,...
6028                                               {3000}
6040    {1921, 1673, 2571, 2575, 2068, 1947, 3362, 272...
Name: movie_id, Length: 1762, dtype: object

In [24]:
len(liked_by_user)

1762

In [25]:
top_10_set = set(top_10)

def precision_at_10(liked_set):
    hits = len(top_10_set & liked_set)
    return hits / 10

In [26]:
scores = liked_by_user.apply(precision_at_10)
scores.mean()

np.float64(0.19324631101021567)

In [27]:
user_item = train.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating",
)

In [28]:
user_item.shape

(5400, 3662)

In [29]:
user_item.iloc[:5, :5]

movie_id,1,2,3,4,5
user_id,,,,,
635,NaN,NaN,NaN,NaN,NaN
636,NaN,NaN,NaN,NaN,NaN
637,5.0,NaN,NaN,NaN,NaN
638,NaN,NaN,NaN,NaN,NaN
639,NaN,NaN,NaN,NaN,NaN


In [30]:
user_item.shape

(5400, 3662)

In [31]:
user_item = user_item.fillna(0)

In [32]:
user_item.iloc[:5, :5]

movie_id,1,2,3,4,5
user_id,,,,,
635,0.0,0.0,0.0,0.0,0.0
636,0.0,0.0,0.0,0.0,0.0
637,5.0,0.0,0.0,0.0,0.0
638,0.0,0.0,0.0,0.0,0.0
639,0.0,0.0,0.0,0.0,0.0


In [33]:
uv pip install scikit-learn

/Library/Frameworks/Python.framework/Versions/3.11/bin/python3.11: No module named uv
Note: you may need to restart the kernel to use updated packages.


In [34]:
from sklearn.metrics.pairwise import cosine_similarity

In [35]:
similarity = cosine_similarity(user_item)

In [36]:
similarity.shape

(5400, 5400)

In [39]:
similarity[:5, :5]

array([[1.        , 0.02669061, 0.08845069, 0.01803662, 0.02469173],
       [0.02669061, 1.        , 0.17466465, 0.17098663, 0.15708817],
       [0.08845069, 0.17466465, 1.        , 0.23531432, 0.23083415],
       [0.01803662, 0.17098663, 0.23531432, 1.        , 0.1701369 ],
       [0.02469173, 0.15708817, 0.23083415, 0.1701369 , 1.        ]])

In [40]:
user_ids = user_item.index
user_ids

Index([ 635,  636,  637,  638,  639,  640,  641,  642,  643,  644,
       ...
       6031, 6032, 6033, 6034, 6035, 6036, 6037, 6038, 6039, 6040],
      dtype='int64', name='user_id', length=5400)

In [41]:
user_ids[0]

np.int64(635)

In [43]:
user_row = similarity[0]
user_row.shape

(5400,)

In [44]:
import numpy as np

similar_positions = np.argsort(user_row)[::-1]
similar_positions[:6]

array([   0, 1020, 4938,  678,  932, 5357])

In [45]:
for pos in similar_positions[1:6]:      # skip position 0 (themselves)
    print(user_ids[pos], round(user_row[pos], 3))

1657 0.224
5579 0.214
1315 0.21
1569 0.197
5998 0.194


In [46]:
neighbour_positions = similar_positions[1:21]     # top 20, excluding self

In [47]:
neighbour_ratings = user_item.iloc[neighbour_positions]
neighbour_ratings.shape

(20, 3662)

In [48]:
film_scores = neighbour_ratings.sum(axis=0)
film_scores.shape

(3662,)

In [49]:
user_635_ratings = user_item.loc[635]
already_seen = user_635_ratings > 0

In [50]:
film_scores[already_seen] = 0

In [51]:
recommendations = film_scores.sort_values(ascending=False).head(10)
recommendations

movie_id
1193    56.0
593     50.0
527     48.0
923     46.0
2028    46.0
912     44.0
1221    42.0
2858    42.0
1213    40.0
919     39.0
dtype: float64

In [52]:
rec_ids = recommendations.index
movies[movies["movie_id"].isin(rec_ids)]

,movie_id,title,genres
523,527,Schindler's List (1993),Drama|War
589,593,"Silence of the Lambs, The (1991)",Drama|Thriller
900,912,Casablanca (1942),Drama|Romance|War
907,919,"Wizard of Oz, The (1939)",Adventure|Children's|Drama|Musical
911,923,Citizen Kane (1941),Drama
1176,1193,One Flew Over the Cuckoo's Nest (1975),Drama
1195,1213,GoodFellas (1990),Crime|Drama
1203,1221,"Godfather: Part II, The (1974)",Action|Crime|Drama
1959,2028,Saving Private Ryan (1998),Action|Drama|War
2789,2858,American Beauty (1999),Comedy|Drama


In [53]:
def recommend_for_user(user_id, k=10):
    # 1. user_id → matrix position
    position = user_item.index.get_loc(user_id)

    # 2. this user's similarity row
    user_row = similarity[position]

    # 3. top-20 neighbour positions, skipping self
    similar_positions = np.argsort(user_row)[::-1]
    neighbour_positions = similar_positions[1:21]

    # 4. pull neighbours' ratings, sum per film
    neighbour_ratings = user_item.iloc[neighbour_positions]
    film_scores = neighbour_ratings.sum(axis=0)

    # 5. zero out films this user already saw
    already_seen = user_item.loc[user_id] > 0
    film_scores[already_seen] = 0

    # 6. return top-k movie IDs
    return film_scores.sort_values(ascending=False).head(k).index.tolist()

In [54]:
recommend_for_user(635)

[1193, 593, 527, 923, 2028, 912, 1221, 2858, 1213, 919]

In [55]:
recs = recommend_for_user(1657)      # 635's closest neighbour, from earlier
movies[movies["movie_id"].isin(recs)]

,movie_id,title,genres
257,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Fantasy|Sci-Fi
523,527,Schindler's List (1993),Drama|War
589,593,"Silence of the Lambs, The (1991)",Drama|Thriller
907,919,"Wizard of Oz, The (1939)",Adventure|Children's|Drama|Musical
1245,1265,Groundhog Day (1993),Comedy|Romance
2286,2355,"Bug's Life, A (1998)",Animation|Children's|Comedy
2326,2395,Rushmore (1998),Comedy
2327,2396,Shakespeare in Love (1998),Comedy|Romance
2789,2858,American Beauty (1999),Comedy|Drama
3045,3114,Toy Story 2 (1999),Animation|Children's|Comedy


In [57]:
item_similarity = cosine_similarity(user_item.T)
item_similarity.shape

(3662, 3662)

In [58]:
item_ids = user_item.columns
item_ids[:5]

Index([1, 2, 3, 4, 5], dtype='int64', name='movie_id')

In [59]:
position = item_ids.get_loc(1)          # film 1's position in the matrix
film_row = item_similarity[position]     # its similarity to all other films

In [60]:
similar_film_positions = np.argsort(film_row)[::-1][1:6]
similar_ids = item_ids[similar_film_positions]
movies[movies["movie_id"].isin(similar_ids)]

,movie_id,title,genres
584,588,Aladdin (1992),Animation|Children's|Comedy|Musical
1245,1265,Groundhog Day (1993),Comedy|Romance
1250,1270,Back to the Future (1985),Comedy|Sci-Fi
2286,2355,"Bug's Life, A (1998)",Animation|Children's|Comedy
3045,3114,Toy Story 2 (1999),Animation|Children's|Comedy


In [61]:
user_635 = user_item.loc[635]
liked_films = user_635[user_635 >= 4].index
liked_films

Index([ 296,  480,  858,  920, 1172, 1203, 1207, 1217, 1251, 1270, 1279, 1286,
       1884, 1952, 1960, 2294, 2303, 2686, 2966, 3035, 3067, 3183, 3198, 3528,
       3614, 3720, 3751, 3911, 3948],
      dtype='int64', name='movie_id')

In [62]:
liked_positions = [item_ids.get_loc(mid) for mid in liked_films]

In [63]:
scores = item_similarity[liked_positions].sum(axis=0)
scores.shape

(3662,)

In [64]:
scores = pd.Series(scores, index=item_ids)

In [65]:
already_seen = user_item.loc[635] > 0
scores[already_seen] = 0

In [66]:
row2_recs = scores.sort_values(ascending=False).head(10)
movies[movies["movie_id"].isin(row2_recs.index)]

,movie_id,title,genres
257,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Fantasy|Sci-Fi
523,527,Schindler's List (1993),Drama|War
589,593,"Silence of the Lambs, The (1991)",Drama|Thriller
604,608,Fargo (1996),Crime|Drama|Thriller
900,912,Casablanca (1942),Drama|Romance|War
907,919,"Wizard of Oz, The (1939)",Adventure|Children's|Drama|Musical
1081,1097,E.T. the Extra-Terrestrial (1982),Children's|Drama|Fantasy|Sci-Fi
1178,1196,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Drama|Sci-Fi|War
1207,1225,Amadeus (1984),Drama
1227,1247,"Graduate, The (1967)",Drama|Romance


In [67]:
def recommend_by_history(user_id, k=10):
    user_ratings = user_item.loc[user_id]
    liked_films = user_ratings[user_ratings >= 4].index

    liked_positions = [item_ids.get_loc(mid) for mid in liked_films]
    scores = item_similarity[liked_positions].sum(axis=0)
    scores = pd.Series(scores, index=item_ids)

    already_seen = user_ratings > 0
    scores[already_seen] = 0

    return scores.sort_values(ascending=False).head(k).index.tolist()

In [69]:
recommend_by_history(635)

[608, 1196, 1247, 919, 1225, 593, 912, 260, 527, 1097]

In [70]:
recommend_by_history(1657)

[1265, 1196, 2396, 608, 296, 2858, 593, 1097, 318, 260]

In [74]:
evaluable_users = liked_by_user.index
in_train = evaluable_users.isin(user_item.index)
scorable_users = evaluable_users[in_train]
len(scorable_users)

1122

In [75]:
def evaluate(recommend_fn, users):
    total = 0
    for user_id in users:
        recs = set(recommend_fn(user_id, k=10))
        liked = liked_by_user[user_id]
        hits = len(recs & liked)
        total += hits / 10
    return total / len(users)

In [79]:
def recommend_popular(user_id, k=10):
    seen = set(user_item.loc[user_id][user_item.loc[user_id] > 0].index)
    recs = [film for film in popularity.index if film not in seen]
    return recs[:k]

In [80]:
p_baseline = evaluate(recommend_popular, scorable_users)
p_history = evaluate(recommend_by_history, scorable_users)
p_neighbours = evaluate(recommend_for_user, scorable_users)

p_baseline, p_history, p_neighbours

(0.199554367201425, 0.2187165775401061, 0.2016934046345802)

In [81]:
import inspect
print(inspect.getsource(recommend_popular))

def recommend_popular(user_id, k=10):
    seen = set(user_item.loc[user_id][user_item.loc[user_id] > 0].index)
    recs = [film for film in popularity.index if film not in seen]
    return recs[:k]



In [82]:
def coverage(recommend_fn, users):
    recommended_films = set()
    for user_id in users:
        recs = recommend_fn(user_id, k=10)
        recommended_films.update(recs)
    return len(recommended_films)

In [83]:
cov_baseline = coverage(recommend_popular, scorable_users)
cov_history = coverage(recommend_by_history, scorable_users)
cov_neighbours = coverage(recommend_for_user, scorable_users)

cov_baseline, cov_history, cov_neighbours

(136, 313, 678)

In [84]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=8, random_state=42, n_init=10)
clusters = kmeans.fit_predict(user_item)

In [85]:
import pandas as pd
pd.Series(clusters).value_counts()

1    2806
3     604
5     538
4     489
0     371
7     284
2     238
6      70
Name: count, dtype: int64

In [86]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=50, random_state=42)
user_factors = svd.fit_transform(user_item)
user_factors.shape

(5400, 50)

In [87]:
svd.explained_variance_ratio_.sum()

np.float64(0.4061428322941553)

In [88]:
kmeans = KMeans(n_clusters=8, random_state=42, n_init=10)
clusters_svd = kmeans.fit_predict(user_factors)

pd.Series(clusters_svd).value_counts()

1    2586
4     684
6     608
5     483
0     318
3     310
2     304
7     107
Name: count, dtype: int64

In [89]:
user_cluster = pd.Series(clusters_svd, index=user_item.index)
user_cluster.head()

user_id
635    1
636    6
637    3
638    6
639    6
dtype: int32

In [90]:
cluster_top_films = {}

for c in range(8):
    members = user_cluster[user_cluster == c].index      # users in tribe c
    tribe_ratings = user_item.loc[members]                # their rating rows
    film_scores = tribe_ratings.sum(axis=0)               # pool: sum per film
    ranked = film_scores.sort_values(ascending=False).index
    cluster_top_films[c] = ranked

In [91]:
movies[movies["movie_id"].isin(cluster_top_films[0][:10])]

,movie_id,title,genres
352,356,Forrest Gump (1994),Comedy|Romance|War
1178,1196,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Drama|Sci-Fi|War
1179,1197,"Princess Bride, The (1987)",Action|Adventure|Comedy|Romance
1245,1265,Groundhog Day (1993),Comedy|Romance
1250,1270,Back to the Future (1985),Comedy|Sci-Fi
1287,1307,When Harry Met Sally... (1989),Comedy|Romance
1899,1968,"Breakfast Club, The (1985)",Comedy|Drama
2647,2716,Ghostbusters (1984),Comedy|Horror
2789,2858,American Beauty (1999),Comedy|Drama
2849,2918,Ferris Bueller's Day Off (1986),Comedy


In [92]:
def recommend_by_persona(user_id, k=10):
    cluster = user_cluster[user_id]                    # which tribe
    ranked = cluster_top_films[cluster]                # tribe's ranked films
    seen = set(user_item.loc[user_id][user_item.loc[user_id] > 0].index)
    recs = [film for film in ranked if film not in seen]
    return recs[:k]

In [93]:
recommend_by_persona(635)

[2858, 2762, 260, 2997, 1210, 2396, 2028, 110, 593, 589]

In [94]:
p_persona = evaluate(recommend_by_persona, scorable_users)
cov_persona = coverage(recommend_by_persona, scorable_users)
p_persona, cov_persona

(0.20053475935828813, 231)

In [95]:
import pickle

bundle = {
    "user_item": user_item,
    "similarity": similarity,
    "item_similarity": item_similarity,
    "item_ids": item_ids,
    "popularity": popularity,
    "movies": movies,
    "user_cluster": user_cluster,
    "cluster_top_films": cluster_top_films,
}

with open("models.pkl", "wb") as f:
    pickle.dump(bundle, f)